In [4]:
"""
================================================================================
Experiment 35: MFCC + CNN + XGBoost + SMOTE + Feature Importance
WITHOUT libROSA (uses python_speech_features + scipy)
FIXED CNN ARCHITECTURE (no negative dimension error)
================================================================================
"""
import os
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import resample
from python_speech_features import mfcc as psf_mfcc
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score,
    balanced_accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, GlobalAveragePooling2D, Input
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
from collections import Counter
import pickle
import gc
import matplotlib.pyplot as plt
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

from imblearn.over_sampling import SMOTE

# =============================================================================
# Configuration - EDIT THESE PATHS
# =============================================================================
SOUND_FILES_PATH = "E:\\FFF\\respiratory-sound-database\\Respiratory_Sound_Database\\Respiratory_Sound_Database\\audio_and_txt_files\\"
DIAGNOSIS_CSV_PATH = "E:\\FFF\\respiratory-sound-database\\Respiratory_Sound_Database\\Respiratory_Sound_Database\\patient_diagnosis.csv"
OUTPUT_DIR = "E:\\Results\\exp35_smote_importance_fixed\\"

MAX_FILES = None  # Set to e.g., 100 for quick test
DTYPE = np.float32
CNN_BATCH_SIZE = 32

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# Feature extraction using python_speech_features (no librosa)
# =============================================================================
def extract_mfcc_from_wav(file_path, num_cep=40, max_pad_len=862, target_sr=16000):
    try:
        sr, audio = wavfile.read(file_path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        if audio.dtype == np.int16:
            audio = audio.astype(np.float32) / 32767.0
        elif audio.dtype == np.int32:
            audio = audio.astype(np.float32) / 2147483648.0
        else:
            audio = audio.astype(np.float32)
        if sr != target_sr:
            number_of_samples = int(round(len(audio) * target_sr / sr))
            audio = resample(audio, number_of_samples)
            sr = target_sr
        target_len = 20 * target_sr
        if len(audio) > target_len:
            audio = audio[:target_len]
        elif len(audio) < target_len:
            repeats = target_len // len(audio) + 1
            audio = np.tile(audio, repeats)[:target_len]
        mfccs = psf_mfcc(audio, samplerate=sr, numcep=num_cep, nfft=512, winlen=0.025, winstep=0.01)
        mfccs = mfccs.T
        if mfccs.shape[1] < max_pad_len:
            pad_width = max_pad_len - mfccs.shape[1]
            mfccs = np.pad(mfccs, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :max_pad_len]
        return mfccs.astype(DTYPE)
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# =============================================================================
# Helper functions (unchanged)
# =============================================================================
def per_class_metrics(y_true, y_pred, class_names):
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=range(len(class_names)), zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    FP = cm.sum(axis=0) - np.diag(cm)
    FN = cm.sum(axis=1) - np.diag(cm)
    TP = np.diag(cm)
    TN = cm.sum() - (FP + FN + TP)
    with np.errstate(divide='ignore', invalid='ignore'):
        FPR = np.divide(FP, FP+TN, where=(FP+TN)>0)
        FNR = np.divide(FN, FN+TP, where=(FN+TP)>0)
        FPR = np.nan_to_num(FPR, nan=0.0)
        FNR = np.nan_to_num(FNR, nan=0.0)
    result = {}
    for i, name in enumerate(class_names):
        result[name] = {'precision':precision[i], 'recall':recall[i], 'f1':f1[i], 'fpr':FPR[i], 'fnr':FNR[i]}
    return result

def plot_feature_importance(importances_list, output_path):
    avg_imp = np.mean(importances_list, axis=0)
    top_idx = np.argsort(avg_imp)[-20:]
    plt.figure(figsize=(10,6))
    plt.barh(range(len(top_idx)), avg_imp[top_idx])
    plt.yticks(range(len(top_idx)), [f"Feature_{i}" for i in top_idx])
    plt.xlabel('Average Importance')
    plt.title('Top 20 CNN-Learned Features by XGBoost Importance')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

# =============================================================================
# Stage 1: Extract raw features
# =============================================================================
print("[Stage 1] Extracting MFCC features (without librosa)...")
all_filenames = [f for f in os.listdir(SOUND_FILES_PATH) if f.endswith('.wav')]
if MAX_FILES is not None:
    all_filenames = all_filenames[:MAX_FILES]

if len(all_filenames) == 0:
    raise FileNotFoundError(f"No .wav files in {SOUND_FILES_PATH}")

print(f"Found {len(all_filenames)} .wav files.")

patient_ids = []
valid_paths = []
for fname in all_filenames:
    try:
        pid = int(fname[:3])
        patient_ids.append(pid)
        valid_paths.append(os.path.join(SOUND_FILES_PATH, fname))
    except:
        print(f"Skipping invalid patient ID filename: {fname}")

diag_df = pd.read_csv(DIAGNOSIS_CSV_PATH, header=None)
diag_dict = dict(zip(diag_df[0].astype(int), diag_df[1]))

labels = [diag_dict.get(pid, None) for pid in patient_ids]
keep = [i for i, lbl in enumerate(labels) if lbl is not None]
patient_ids = [patient_ids[i] for i in keep]
valid_paths = [valid_paths[i] for i in keep]
labels = [labels[i] for i in keep]
print(f"After matching diagnosis: {len(valid_paths)} files remain.")

features_list = []
for i, fp in enumerate(valid_paths):
    feat = extract_mfcc_from_wav(fp)
    features_list.append(feat)
    if (i+1) % 50 == 0:
        print(f"  Processed {i+1}/{len(valid_paths)} files")
    gc.collect()

success_idx = [i for i, f in enumerate(features_list) if f is not None]
features = np.array([features_list[i] for i in success_idx], dtype=DTYPE)
patient_ids = [patient_ids[i] for i in success_idx]
labels = [labels[i] for i in success_idx]
print(f"Successfully extracted features from {len(features)} files.")
if len(features) == 0:
    raise RuntimeError("No features extracted.")

# =============================================================================
# Stage 2: Clean data
# =============================================================================
print("\n[Stage 2] Cleaning data...")
new_features, new_labels, new_pids = [], [], []
for f, lbl, pid in zip(features, labels, patient_ids):
    if lbl in ['Asthma', 'LRTI']:
        continue
    elif lbl in ['Healthy', 'URTI']:
        new_labels.append('Other')
    else:
        new_labels.append(lbl)
    new_features.append(f)
    new_pids.append(pid)

if len(new_features) == 0:
    raise RuntimeError("No samples left after cleaning.")
features = np.array(new_features, dtype=DTYPE)
labels = new_labels
patient_ids = new_pids
print(f"Remaining samples: {len(features)}, unique patients: {len(set(patient_ids))}")
print("Class distribution:", Counter(labels))

features = features[..., np.newaxis]  # (n, 40, 862, 1)

cleaned_file = os.path.join(OUTPUT_DIR, "cleaned_data.pkl")
with open(cleaned_file, 'wb') as f:
    pickle.dump({'features':features, 'labels':labels, 'patient_ids':patient_ids}, f)

# =============================================================================
# Stage 3: Cross-validation with SMOTE and fixed CNN
# =============================================================================
print("\n[Stage 3] Patient-level cross-validation...")
df = pd.DataFrame({'features': list(features), 'label': labels, 'patient_id': patient_ids})
unique_patients = df.groupby('patient_id')['label'].first().reset_index()
n_unique = len(unique_patients)
print(f"Unique patients for CV: {n_unique}")
if n_unique < 2:
    raise ValueError(f"Need at least 2 unique patients, got {n_unique}.")

le_global = LabelEncoder()
unique_patients['label_enc'] = le_global.fit_transform(unique_patients['label'])

n_splits = 3
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

fold_results = []
all_feature_importances = []
all_y_test, all_y_pred = [], []

for fold, (train_idx, test_idx) in enumerate(skf.split(unique_patients['patient_id'], unique_patients['label_enc'])):
    fold_num = fold + 1
    print(f"\n--- Fold {fold_num}/{n_splits} ---")
    train_patients = unique_patients.iloc[train_idx]['patient_id'].values
    test_patients = unique_patients.iloc[test_idx]['patient_id'].values
    
    train_df = df[df['patient_id'].isin(train_patients)]
    test_df = df[df['patient_id'].isin(test_patients)]
    
    X_train = np.stack(train_df['features'].values).astype(DTYPE)
    X_test = np.stack(test_df['features'].values).astype(DTYPE)
    y_train_str = train_df['label'].values
    y_test_str = test_df['label'].values
    
    le_fold = LabelEncoder()
    y_train = le_fold.fit_transform(y_train_str)
    y_test = le_fold.transform(y_test_str)
    
    # ================ FIXED CNN ARCHITECTURE ================
    # Using 3 Conv+Pool layers with padding='same' to avoid dimension collapse
    cnn = Sequential([
        Input(shape=(40, 862, 1)),
        Conv2D(16, (2,2), padding='same', activation='relu'),
        MaxPooling2D(pool_size=(2,2)),
        Dropout(0.2),
        Conv2D(32, (2,2), padding='same', activation='relu'),
        MaxPooling2D(pool_size=(2,2)),
        Dropout(0.2),
        Conv2D(64, (2,2), padding='same', activation='relu'),
        GlobalAveragePooling2D()  # replaces third MaxPooling + extra Dropout
    ])
    cnn.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    X_train_feat = cnn.predict(X_train, batch_size=CNN_BATCH_SIZE, verbose=0)
    X_test_feat = cnn.predict(X_test, batch_size=CNN_BATCH_SIZE, verbose=0)
    X_train_feat = X_train_feat.reshape(X_train_feat.shape[0], -1).astype(DTYPE)
    X_test_feat = X_test_feat.reshape(X_test_feat.shape[0], -1).astype(DTYPE)
    
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train_feat, y_train)
    print(f"After SMOTE: train size {len(X_train_res)}")
    
    xgb_model = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.8, gamma=0.1,
        eval_metric='mlogloss', random_state=42
    )
    xgb_model.fit(X_train_res, y_train_res)
    y_pred = xgb_model.predict(X_test_feat)
    
    imp = xgb_model.feature_importances_
    all_feature_importances.append(imp)
    imp_df = pd.DataFrame({'feature_index': range(len(imp)), 'importance': imp})
    imp_df.sort_values('importance', ascending=False).to_csv(
        os.path.join(OUTPUT_DIR, f"feature_importance_fold{fold_num}.csv"), index=False)
    
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    macro_rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    w_f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    w_rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    FP = cm.sum(axis=0) - np.diag(cm)
    FN = cm.sum(axis=1) - np.diag(cm)
    TP = np.diag(cm)
    TN = cm.sum() - (FP+FN+TP)
    with np.errstate(divide='ignore', invalid='ignore'):
        fpr = np.mean(np.divide(FP, FP+TN, where=(FP+TN)>0))
        fnr = np.mean(np.divide(FN, FN+TP, where=(FN+TP)>0))
    fpr = np.nan_to_num(fpr, nan=0.0)
    fnr = np.nan_to_num(fnr, nan=0.0)
    
    fold_results.append({
        'fold': fold_num,
        'accuracy': acc,
        'macro_f1': macro_f1,
        'macro_recall': macro_rec,
        'weighted_f1': w_f1,
        'weighted_recall': w_rec,
        'balanced_accuracy': bal_acc,
        'fpr': fpr,
        'fnr': fnr
    })
    all_y_test.extend(y_test)
    all_y_pred.extend(y_pred)
    pd.DataFrame(fold_results).to_csv(os.path.join(OUTPUT_DIR, "fold_results.csv"), index=False)
    print(f"Fold {fold_num} accuracy: {acc:.4f}")

# Aggregate feature importance
if all_feature_importances:
    avg_imp = np.mean(all_feature_importances, axis=0)
    std_imp = np.std(all_feature_importances, axis=0)
    imp_summary = pd.DataFrame({'feature_index': range(len(avg_imp)), 'importance_mean': avg_imp, 'importance_std': std_imp})
    imp_summary.sort_values('importance_mean', ascending=False).to_csv(
        os.path.join(OUTPUT_DIR, "feature_importance_summary.csv"), index=False)
    plot_feature_importance(all_feature_importances, os.path.join(OUTPUT_DIR, "feature_importance_plot.png"))

pd.DataFrame({'y_true': all_y_test, 'y_pred': all_y_pred}).to_csv(os.path.join(OUTPUT_DIR, "predictions.csv"), index=False)

# =============================================================================
# Stage 4: Final metrics
# =============================================================================
print("\n[Stage 4] Final aggregation...")
fold_df = pd.read_csv(os.path.join(OUTPUT_DIR, "fold_results.csv"))
avg = {}
for m in ['accuracy','macro_f1','macro_recall','weighted_f1','weighted_recall','balanced_accuracy','fpr','fnr']:
    avg[m] = fold_df[m].mean()
    avg[f'{m}_std'] = fold_df[m].std()

inv_fpr = 1 - avg['fpr']
weights = {'macro_f1':0.249, 'macro_recall':0.249, 'weighted_f1':0.157, 'weighted_recall':0.096,
           'accuracy':0.096, 'balanced_accuracy':0.058, 'inverted_fpr':0.058, 'fnr':0.037}
mcdm = (avg['macro_f1']*weights['macro_f1'] + avg['macro_recall']*weights['macro_recall'] +
        avg['weighted_f1']*weights['weighted_f1'] + avg['weighted_recall']*weights['weighted_recall'] +
        avg['accuracy']*weights['accuracy'] + avg['balanced_accuracy']*weights['balanced_accuracy'] +
        inv_fpr*weights['inverted_fpr'] + avg['fnr']*weights['fnr'])

print("\n" + "="*50)
print("FINAL AVERAGED METRICS ({} folds)".format(len(fold_df)))
for m in ['accuracy','macro_f1','macro_recall','weighted_f1','weighted_recall','balanced_accuracy','fpr','fnr']:
    print(f"  {m:18s}: {avg[m]:.4f} ± {avg[f'{m}_std']:.4f}")
print(f"  inverted_fpr        : {inv_fpr:.4f}")
print(f"  MCDM SCORE          : {mcdm:.4f}")
print("="*50)

summary = pd.DataFrame({
    'metric': ['accuracy','macro_f1','macro_recall','weighted_f1','weighted_recall','balanced_accuracy','fpr','fnr','mcdm_score'],
    'mean': [avg['accuracy'], avg['macro_f1'], avg['macro_recall'],
             avg['weighted_f1'], avg['weighted_recall'],
             avg['balanced_accuracy'], avg['fpr'], avg['fnr'], mcdm],
    'std': [avg['accuracy_std'], avg['macro_f1_std'], avg['macro_recall_std'],
            avg['weighted_f1_std'], avg['weighted_recall_std'],
            avg['balanced_accuracy_std'], avg['fpr_std'], avg['fnr_std'], np.nan]
})
summary.to_csv(os.path.join(OUTPUT_DIR, "final_summary.csv"), index=False)
print(f"\nAll results saved to {OUTPUT_DIR}")

[Stage 1] Extracting MFCC features (without librosa)...
Found 920 .wav files.
After matching diagnosis: 920 files remain.
  Processed 50/920 files
  Processed 100/920 files
  Processed 150/920 files
  Processed 200/920 files
  Processed 250/920 files
  Processed 300/920 files
  Processed 350/920 files
  Processed 400/920 files
  Processed 450/920 files
  Processed 500/920 files
  Processed 550/920 files
  Processed 600/920 files
  Processed 650/920 files
  Processed 700/920 files
  Processed 750/920 files
  Processed 800/920 files
  Processed 850/920 files
  Processed 900/920 files
Successfully extracted features from 920 files.

[Stage 2] Cleaning data...
Remaining samples: 917, unique patients: 123
Class distribution: Counter({'COPD': 793, 'Other': 58, 'Pneumonia': 37, 'Bronchiectasis': 16, 'Bronchiolitis': 13})

[Stage 3] Patient-level cross-validation...
Unique patients for CV: 123

--- Fold 1/3 ---
After SMOTE: train size 2885
Fold 1 accuracy: 0.8168

--- Fold 2/3 ---
After SMOTE: